In [76]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [77]:
import os

# Define your project directory path
PROJECT_PATH = '/content/drive/MyDrive/Implied-Volatility-project'

# Change the current working directory to your project folder
os.chdir(PROJECT_PATH)

# Verify you are in the right place
print("Current Working Directory:", os.getcwd())

Current Working Directory: /content/drive/MyDrive/Implied-Volatility-project


In [103]:
# =============================================================================
# SECTION 0 — CONFIG
# =============================================================================
import warnings
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from sklearn.metrics import mean_squared_error
warnings.filterwarnings("ignore")

FEATURES_OBS_PATH  = "features_observed.parquet"
FEATURES_PRED_PATH = "features_predict.parquet"
SUBMISSION_PATH    = "submission_spline.csv"
OOF_PATH           = "oof_spline.parquet"

DATETIME_COL   = "datetime"
TARGET_COL     = "iv"
STRIKE_COL     = "strike"
SYMBOL_COL     = "symbol"
OPTION_TYPE_COL= "option_type"
DTE_COL        = "dte_days"

# Spline fallback: if a slice has fewer than this many points, use linear
MIN_POINTS_CUBIC = 4

# CV: number of chronological folds
N_FOLDS = 5

In [104]:
# =============================================================================
# SECTION 1 — LOAD DATA
# =============================================================================

obs_df  = pd.read_parquet(FEATURES_OBS_PATH)
pred_df = pd.read_parquet(FEATURES_PRED_PATH)

obs_df[DATETIME_COL]  = pd.to_datetime(obs_df[DATETIME_COL])
pred_df[DATETIME_COL] = pd.to_datetime(pred_df[DATETIME_COL])

print(f"[LOAD] observed  : {obs_df.shape}")
print(f"[LOAD] predict   : {pred_df.shape}")
print(f"\n[LOAD] Observed DTE distribution:")
print(obs_df[DTE_COL].value_counts().sort_index())
print(f"\n[LOAD] Predict DTE distribution:")
print(pred_df[DTE_COL].value_counts().sort_index())

[LOAD] observed  : (21840, 46)
[LOAD] predict   : (5460, 46)

[LOAD] Observed DTE distribution:
dte_days
0     1692
3     1690
4     1670
5     1699
6     1685
7     1676
10    1662
12    1703
13    1676
14    1642
17    1689
18    1643
19    1713
Name: count, dtype: int64

[LOAD] Predict DTE distribution:
dte_days
0     408
3     410
4     430
5     401
6     415
7     424
10    438
12    397
13    424
14    458
17    411
18    457
19    387
Name: count, dtype: int64


In [105]:
# =============================================================================
# SECTION 2 — DEFINE FEATURE COLUMNS
# (Spline uses strike as the only "feature" — no ML feature matrix needed)
# =============================================================================

# Verify required columns exist in both dataframes
required_cols = [DATETIME_COL, TARGET_COL, STRIKE_COL, SYMBOL_COL, OPTION_TYPE_COL, DTE_COL]
for col in required_cols:
    assert col in obs_df.columns,  f"Missing in obs_df:  {col}"
    assert col in pred_df.columns, f"Missing in pred_df: {col}"

print("[COLUMNS] All required columns present.")
print(f"\n[COLUMNS] Unique strikes in observed : {obs_df[STRIKE_COL].nunique()}")
print(f"[COLUMNS] Unique strikes in predict  : {pred_df[STRIKE_COL].nunique()}")
print(f"\n[COLUMNS] Strike range observed : {obs_df[STRIKE_COL].min()} – {obs_df[STRIKE_COL].max()}")
print(f"[COLUMNS] Strike range predict  : {pred_df[STRIKE_COL].min()} – {pred_df[STRIKE_COL].max()}")

# Check all predict strikes are within observed range (extrapolation check)
pred_strikes    = set(pred_df[STRIKE_COL].unique())
obs_strikes     = set(obs_df[STRIKE_COL].unique())
unseen_strikes  = pred_strikes - obs_strikes
print(f"\n[COLUMNS] Predict strikes not seen in training: {len(unseen_strikes)}")
if unseen_strikes:
    print(f"           {sorted(unseen_strikes)}")

[COLUMNS] All required columns present.

[COLUMNS] Unique strikes in observed : 28
[COLUMNS] Unique strikes in predict  : 28

[COLUMNS] Strike range observed : 23800 – 26500
[COLUMNS] Strike range predict  : 23800 – 26500

[COLUMNS] Predict strikes not seen in training: 0


In [106]:
# =============================================================================
# SECTION 3 — PURGED WALK-FORWARD CROSS-VALIDATION SETUP
# For spline, CV works as follows:
#   - Train slice: observed strikes at timestamp t (used to fit spline)
#   - Val slice  : held-out strikes at same timestamp t (used to score)
# We hold out 1-2 strikes per smile as pseudo-missing to simulate Kaggle task.
# Time purge ensures val timestamps are strictly after train timestamps.
# =============================================================================

def purged_walk_forward_splits(
    df      : pd.DataFrame,
    n_folds : int,
    purge   : int = 2,
) -> list[tuple]:
    """
    Returns list of (train_timestamps, val_timestamps) pairs.
    Each is a list of datetime values, not row indices.
    """
    all_times = np.sort(df[DATETIME_COL].unique())
    T         = len(all_times)
    fold_size = T // (n_folds + 1)
    splits    = []

    for fold in range(n_folds):
        purge_cutoff  = all_times[max(0, (fold + 1) * fold_size - purge)]
        val_start     = all_times[(fold + 1) * fold_size]
        val_end       = all_times[min((fold + 2) * fold_size - 1, T - 1)]

        train_times   = all_times[all_times < purge_cutoff]
        val_times     = all_times[(all_times >= val_start) & (all_times <= val_end)]

        if len(train_times) == 0 or len(val_times) == 0:
            continue

        splits.append((train_times, val_times))
        print(f"[CV] Fold {fold+1}: train={len(train_times):,} timestamps | "
              f"purge={purge} | val={len(val_times):,} timestamps")

    return splits


cv_splits = purged_walk_forward_splits(obs_df, N_FOLDS)

[CV] Fold 1: train=160 timestamps | purge=2 | val=162 timestamps
[CV] Fold 2: train=322 timestamps | purge=2 | val=162 timestamps
[CV] Fold 3: train=484 timestamps | purge=2 | val=162 timestamps
[CV] Fold 4: train=646 timestamps | purge=2 | val=162 timestamps
[CV] Fold 5: train=808 timestamps | purge=2 | val=162 timestamps


In [183]:
# =============================================================================
# SECTION 4 — CROSS-VALIDATION TRAINING
# For each val timestamp, we:
#   1. Take all observed strikes at that timestamp
#   2. Hold out 20% of strikes as pseudo-missing
#   3. Fit cubic spline on remaining 80%
#   4. Predict held-out strikes and score MSE
# =============================================================================

def fit_predict_spline(
    known_strikes : np.ndarray,
    known_ivs     : np.ndarray,
    query_strikes : np.ndarray,
) -> np.ndarray:
    sort_idx      = np.argsort(known_strikes)
    known_strikes = known_strikes[sort_idx]
    known_ivs     = known_ivs[sort_idx]

    if len(known_strikes) >= MIN_POINTS_CUBIC:
        cs    = CubicSpline(known_strikes, known_ivs, bc_type="natural")
        preds = cs(query_strikes)
    else:
        preds = np.interp(query_strikes, known_strikes, known_ivs)

    return preds


def run_cv_spline(
    obs_df   : pd.DataFrame,
    splits   : list[tuple],
    holdout  : float = 0.2,
    seed     : int   = 42,
) -> tuple[pd.DataFrame, list[float]]:
    """
    For each fold, iterate over val timestamps.
    At each timestamp, per (option_type):
      - randomly hold out `holdout` fraction of strikes
      - fit spline on remaining strikes
      - predict held-out strikes
    """
    rng      = np.random.default_rng(seed)
    oof_rows = []
    fold_mses= []

    for fold_i, (train_times, val_times) in enumerate(splits, 1):
        fold_preds = []
        fold_trues = []

        val_df = obs_df[obs_df[DATETIME_COL].isin(val_times)]

        for ts, ts_group in val_df.groupby(DATETIME_COL):
            for otype, otype_group in ts_group.groupby(OPTION_TYPE_COL):
                otype_group = otype_group.sort_values(STRIKE_COL)
                strikes     = otype_group[STRIKE_COL].values
                ivs         = otype_group[TARGET_COL].values
                n           = len(strikes)

                if n < 3:
                    continue

                # Hold out random subset as pseudo-missing
                n_holdout  = max(1, int(n * holdout))
                holdout_idx= rng.choice(n, size=n_holdout, replace=False)
                train_mask = np.ones(n, dtype=bool)
                train_mask[holdout_idx] = False

                known_s = strikes[train_mask]
                known_v = ivs[train_mask]
                query_s = strikes[holdout_idx]
                true_v  = ivs[holdout_idx]

                pred_v = fit_predict_spline(known_s, known_v, query_s)
                pred_v  = np.clip(pred_v, 0.01, 5.0)

                fold_preds.extend(pred_v.tolist())
                fold_trues.extend(true_v.tolist())

                for sym, p, t in zip(
                    otype_group.iloc[holdout_idx][SYMBOL_COL].values,
                    pred_v, true_v
                ):
                    oof_rows.append({
                        DATETIME_COL : ts,
                        SYMBOL_COL   : sym,
                        "oof_pred"   : p,
                        TARGET_COL   : t,
                        "fold"       : fold_i,
                    })

        fold_mse = mean_squared_error(fold_trues, fold_preds)
        fold_mses.append(fold_mse)
        print(f"[CV] Fold {fold_i} → MSE: {fold_mse:.8f}  RMSE: {fold_mse**0.5:.8f}")

    oof_df = pd.DataFrame(oof_rows)
    overall_mse = mean_squared_error(oof_df[TARGET_COL], oof_df["oof_pred"])
    print(f"\n[CV] OOF MSE  (pooled) : {overall_mse:.8f}")
    print(f"[CV] OOF RMSE (pooled) : {overall_mse**0.5:.8f}")

    return oof_df, fold_mses


oof_df, fold_mses = run_cv_spline(obs_df, cv_splits)

[CV] Fold 1 → MSE: 0.00000500  RMSE: 0.00223639
[CV] Fold 2 → MSE: 0.00000693  RMSE: 0.00263181
[CV] Fold 3 → MSE: 0.00000311  RMSE: 0.00176444
[CV] Fold 4 → MSE: 0.00000186  RMSE: 0.00136313
[CV] Fold 5 → MSE: 0.00026173  RMSE: 0.01617814

[CV] OOF MSE  (pooled) : 0.00005634
[CV] OOF RMSE (pooled) : 0.00750606


In [184]:
# =============================================================================
# SECTION 5 — FEATURE IMPORTANCE
# For spline, "importance" = per-DTE and per-option_type MSE breakdown.
# This replaces gain-based importance and tells us where the spline struggles.
# =============================================================================

def spline_importance(oof_df: pd.DataFrame, obs_df: pd.DataFrame) -> None:
    """
    Break down OOF MSE by DTE and option_type to identify
    which slices drive error — equivalent of feature importance
    for the spline model.
    """
    # Merge DTE back onto oof_df
    meta = obs_df[[SYMBOL_COL, DATETIME_COL, DTE_COL, OPTION_TYPE_COL]].drop_duplicates()
    oof  = oof_df.merge(meta, on=[SYMBOL_COL, DATETIME_COL], how="left")

    print("[IMPORTANCE] MSE breakdown by DTE:")
    dte_breakdown = (
        oof.groupby(DTE_COL)
           .apply(lambda g: mean_squared_error(g[TARGET_COL], g["oof_pred"]))
           .rename("mse")
           .reset_index()
    )
    dte_breakdown["rmse"] = dte_breakdown["mse"] ** 0.5
    print(dte_breakdown.sort_values("mse", ascending=False).to_string(index=False))

    print("\n[IMPORTANCE] MSE breakdown by option_type:")
    otype_breakdown = (
        oof.groupby(OPTION_TYPE_COL)
           .apply(lambda g: mean_squared_error(g[TARGET_COL], g["oof_pred"]))
           .rename("mse")
           .reset_index()
    )
    otype_breakdown["rmse"] = otype_breakdown["mse"] ** 0.5
    print(otype_breakdown.to_string(index=False))


spline_importance(oof_df, obs_df)

[IMPORTANCE] MSE breakdown by DTE:
 dte_days      mse     rmse
        0 0.000575 0.023989
       17 0.000009 0.002974
       12 0.000008 0.002894
       13 0.000007 0.002567
        3 0.000006 0.002448
       10 0.000004 0.002066
        6 0.000003 0.001745
        4 0.000003 0.001670
       14 0.000002 0.001428
        7 0.000001 0.001185
        5 0.000001 0.001139

[IMPORTANCE] MSE breakdown by option_type:
option_type      mse     rmse
         CE 0.000072 0.008514
         PE 0.000040 0.006321


In [185]:
# =============================================================================
# SECTION 6 — RETRAIN ON FULL OBSERVED DATA
# No actual "retraining" needed — spline is fit fresh per timestamp at
# inference time using ALL observed strikes at that timestamp.
# This section validates the final spline has full data available.
# =============================================================================

def validate_full_coverage(
    obs_df  : pd.DataFrame,
    pred_df : pd.DataFrame,
) -> pd.DataFrame:
    """
    For each (datetime, option_type) in pred_df, confirm that obs_df
    has at least MIN_POINTS_CUBIC observed strikes available to fit the spline.
    Flags any slices that will fall back to linear interpolation.
    """
    pred_slices = pred_df.groupby([DATETIME_COL, OPTION_TYPE_COL]).size().reset_index()
    obs_counts  = (
        obs_df.groupby([DATETIME_COL, OPTION_TYPE_COL])[STRIKE_COL]
              .count()
              .rename("n_observed")
              .reset_index()
    )

    coverage = pred_slices.merge(
        obs_counts, on=[DATETIME_COL, OPTION_TYPE_COL], how="left"
    )
    coverage["n_observed"]   = coverage["n_observed"].fillna(0).astype(int)
    coverage["spline_mode"]  = np.where(
        coverage["n_observed"] >= MIN_POINTS_CUBIC, "cubic", "linear"
    )

    cubic_pct = (coverage["spline_mode"] == "cubic").mean() * 100
    print(f"[COVERAGE] Total predict slices    : {len(coverage):,}")
    print(f"[COVERAGE] Cubic spline eligible   : {cubic_pct:.1f}%")
    print(f"[COVERAGE] Linear fallback slices  : {(coverage['spline_mode']=='linear').sum()}")

    problem_slices = coverage[coverage["n_observed"] < 2]
    if len(problem_slices) > 0:
        print(f"\n[COVERAGE] ⚠ {len(problem_slices)} slices have <2 observed points:")
        print(problem_slices.to_string(index=False))
    else:
        print(f"\n[COVERAGE] ✓ All slices have sufficient observed points.")

    return coverage


coverage_df = validate_full_coverage(obs_df, pred_df)

[COVERAGE] Total predict slices    : 1,862
[COVERAGE] Cubic spline eligible   : 100.0%
[COVERAGE] Linear fallback slices  : 0

[COVERAGE] ✓ All slices have sufficient observed points.


In [209]:
# =============================================================================
# SECTION 7 — GENERATE SUBMISSION
# For each (datetime, option_type) slice:
#   1. Grab all observed strikes + IVs at that timestamp from obs_df
#   2. Fit cubic spline
#   3. Predict missing strikes from pred_df
# =============================================================================

def generate_submission(
    obs_df   : pd.DataFrame,
    pred_df  : pd.DataFrame,
    out_path : str,
) -> pd.DataFrame:

    results = []

    for (ts, otype), pred_group in pred_df.groupby([DATETIME_COL, OPTION_TYPE_COL]):

        obs_slice = obs_df[
            (obs_df[DATETIME_COL]    == ts) &
            (obs_df[OPTION_TYPE_COL] == otype)
        ].sort_values(STRIKE_COL)

        known_strikes = obs_slice[STRIKE_COL].values
        known_ivs     = obs_slice[TARGET_COL].values
        query_strikes = pred_group[STRIKE_COL].values
        dte           = pred_group[DTE_COL].iloc[0]

        if dte == 0:
            query_strikes_clamped = np.clip(query_strikes, known_strikes.min(), known_strikes.max())
            pred_ivs = fit_predict_spline(known_strikes, known_ivs, query_strikes_clamped)
            pred_ivs = np.clip(pred_ivs, 0.01, known_ivs.max() * 1.2)
        else:
            pred_ivs = fit_predict_spline(known_strikes, known_ivs, query_strikes)
            pred_ivs = np.clip(pred_ivs, 0.01, 5.0)

        for sym, iv in zip(pred_group[SYMBOL_COL].values, pred_ivs):
            results.append({SYMBOL_COL: sym, DATETIME_COL: ts, TARGET_COL: iv})

    submission = pd.DataFrame(results)
    submission["id"] = (
        submission[DATETIME_COL].dt.strftime("%d-%m-%Y %H:%M")
        + "||" +
        submission[SYMBOL_COL]
    )
    submission[TARGET_COL] = submission[TARGET_COL].round(6)
    submission = submission[["id", TARGET_COL]].sort_values("id").reset_index(drop=True)
    submission.to_csv(out_path, index=False)
    print(f"[SUBMISSION] {len(submission):,} rows saved → {out_path}")
    print(submission.head(5).to_string(index=False))
    return submission


submission_df = generate_submission(obs_df, pred_df, SUBMISSION_PATH)

[SUBMISSION] 5,460 rows saved → submission_spline.csv
                                   id       iv
07-01-2026 09:15||NIFTY27JAN2624100PE 0.163989
07-01-2026 09:15||NIFTY27JAN2625500CE 0.113259
07-01-2026 09:15||NIFTY27JAN2625800CE 0.100904
07-01-2026 09:20||NIFTY27JAN2624000PE 0.169678
07-01-2026 09:20||NIFTY27JAN2624200PE 0.158913


In [210]:
# =============================================================================
# SECTION 8 — SAVE ARTIFACTS
# =============================================================================

oof_df.to_parquet(OOF_PATH, index=False)
coverage_df.to_csv("coverage_report.csv", index=False)

print(f"[SAVE] OOF predictions  → {OOF_PATH}")
print(f"[SAVE] Coverage report  → coverage_report.csv")
print(f"[SAVE] Submission       → {SUBMISSION_PATH}")

[SAVE] OOF predictions  → oof_spline.parquet
[SAVE] Coverage report  → coverage_report.csv
[SAVE] Submission       → submission_spline.csv


In [211]:
# =============================================================================
# SECTION 9 — FINAL VALIDATION REPORT
# =============================================================================

def final_validation_report(
    oof_df    : pd.DataFrame,
    fold_mses : list[float],
    obs_df    : pd.DataFrame,
) -> None:

    print("\n" + "="*60)
    print("FINAL VALIDATION REPORT — CUBIC SPLINE MODEL")
    print("="*60)

    # Per-fold metrics
    fold_summary = pd.DataFrame({
        "fold"    : range(1, len(fold_mses) + 1),
        "val_mse" : [round(m, 8) for m in fold_mses],
        "val_rmse": [round(m**0.5, 8) for m in fold_mses],
    })
    print("\n▸ Per-fold metrics:")
    print(fold_summary.to_string(index=False))

    # Overall OOF
    overall_mse = mean_squared_error(oof_df[TARGET_COL], oof_df["oof_pred"])
    print(f"\n▸ OOF MSE  (pooled) : {overall_mse:.8f}")
    print(f"▸ OOF RMSE (pooled) : {overall_mse**0.5:.8f}")

    # DTE breakdown
    meta = obs_df[[SYMBOL_COL, DATETIME_COL, DTE_COL]].drop_duplicates()
    oof  = oof_df.merge(meta, on=[SYMBOL_COL, DATETIME_COL], how="left")
    print("\n▸ MSE by DTE (worst first):")
    dte_mse = (
        oof.groupby(DTE_COL)
           .apply(lambda g: mean_squared_error(g[TARGET_COL], g["oof_pred"]))
           .rename("mse").reset_index()
    )
    dte_mse["rmse"] = dte_mse["mse"] ** 0.5
    print(dte_mse.sort_values("mse", ascending=False).to_string(index=False))

    # Submission stats
    print(f"\n▸ Submission IV distribution:")
    print(f"  mean : {submission_df['iv'].mean():.6f}")
    print(f"  std  : {submission_df['iv'].std():.6f}")
    print(f"  min  : {submission_df['iv'].min():.6f}")
    print(f"  max  : {submission_df['iv'].max():.6f}")
    print("="*60)


final_validation_report(oof_df, fold_mses, obs_df)


FINAL VALIDATION REPORT — CUBIC SPLINE MODEL

▸ Per-fold metrics:
 fold  val_mse  val_rmse
    1 0.000005  0.002236
    2 0.000007  0.002632
    3 0.000003  0.001764
    4 0.000002  0.001363
    5 0.000262  0.016178

▸ OOF MSE  (pooled) : 0.00005634
▸ OOF RMSE (pooled) : 0.00750606

▸ MSE by DTE (worst first):
 dte_days      mse     rmse
        0 0.000575 0.023989
       17 0.000009 0.002974
       12 0.000008 0.002894
       13 0.000007 0.002567
        3 0.000006 0.002448
       10 0.000004 0.002066
        6 0.000003 0.001745
        4 0.000003 0.001670
       14 0.000002 0.001428
        7 0.000001 0.001185
        5 0.000001 0.001139

▸ Submission IV distribution:
  mean : 0.185632
  std  : 0.243923
  min  : 0.018310
  max  : 5.384760


In [213]:
print(pred_df[DATETIME_COL].nunique())
print(obs_df[DATETIME_COL].nunique())
print(pred_df[DATETIME_COL].min(), pred_df[DATETIME_COL].max())

972
975
2026-01-07 09:15:00 2026-01-27 15:25:00
